In [78]:
!pip install -q pandas plotly


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [79]:
import pandas as pd
import glob
import json
import plotly.express as px

def load_data(paths):
    files = []
    for p in paths:
        files.extend(glob.glob(f"{p}/**/result.json", recursive=True))
    df = pd.DataFrame([json.load(open(f)) for f in files])

    df["nonorm"] = df["REF"] / df["NONORM"]
    df["refref"] = df["REF"] / df["REFREF"]
    df["speedup_pct"] = (100 * df["REF"] / df["COUPLED"] - 100).round(2)

    df = df.round(3)
    df = df.sort_values(by=['model_id', 'prompt_len', 'batch_size', 'new_tokens'])
    return df

paths = [
    './multirun/2026-03-17/sweep-decode-qwen-1-7',
    './multirun/2026-03-17/sweep-decode-no-qwen-1-7',
    './multirun/2026-03-17/sweep-train',
    './multirun/2026-03-17/llama-8b-decode'
]
df = load_data(paths)
df

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
44,HuggingFaceTB/SmolLM2-135M-Instruct,5382,1,1,0.018,0.017,0.014,0.018,1.059,1.235,1.004,5.94
36,HuggingFaceTB/SmolLM2-135M-Instruct,5382,32,1,0.213,0.189,0.170,0.212,1.127,1.249,1.003,12.74
42,HuggingFaceTB/SmolLM2-135M-Instruct,5382,64,1,0.455,0.430,0.372,0.454,1.058,1.224,1.001,5.80
51,HuggingFaceTB/SmolLM2-135M-Instruct,5382,128,1,0.913,0.866,0.680,0.850,1.055,1.342,1.074,5.49
18,Qwen/Qwen3-0.6B,20,1,1,0.014,0.013,0.011,0.013,1.082,1.216,1.019,8.22
...,...,...,...,...,...,...,...,...,...,...,...,...
10,meta-llama/Llama-3.2-3B,20,256,256,14.065,13.718,12.849,14.091,1.025,1.095,0.998,2.54
47,meta-llama/Llama-3.2-3B,5382,1,1,0.077,0.075,0.071,0.076,1.027,1.088,1.014,2.68
55,meta-llama/Llama-3.2-3B,5382,32,1,1.800,1.744,1.610,1.800,1.032,1.118,1.000,3.20
53,meta-llama/Llama-3.2-3B,5382,64,1,3.583,3.474,3.203,3.584,1.031,1.118,1.000,3.14


In [80]:
# On device decode with BS=1 new_tokens=256
df[(df["batch_size"] == 1) & (df["new_tokens"] == 256)]

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
22,Qwen/Qwen3-0.6B,20,1,256,3.029,2.815,2.529,3.024,1.076,1.197,1.002,7.60
1,Qwen/Qwen3-1.7B,20,1,256,2.139,1.998,1.800,2.184,1.070,1.188,0.979,7.02
57,meta-llama/Llama-3.1-8B,20,1,256,6.945,6.462,5.900,6.831,1.075,1.177,1.017,7.48
12,meta-llama/Llama-3.2-1B,20,1,256,2.062,1.931,1.751,2.066,1.068,1.178,0.998,6.77
28,meta-llama/Llama-3.2-3B,20,1,256,4.221,3.947,3.615,4.320,1.069,1.168,0.977,6.92


In [81]:
# Large scale decode with BS=256 new_tokens=256
df[(df["batch_size"] == 256) & (df["new_tokens"] == 256)]

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
26,Qwen/Qwen3-0.6B,20,256,256,11.518,11.267,10.820,11.514,1.022,1.065,1.000,2.23
3,Qwen/Qwen3-1.7B,20,256,256,16.234,15.942,22.716,24.390,1.018,0.715,0.666,1.83
60,meta-llama/Llama-3.1-8B,20,256,256,18.251,17.910,16.924,18.318,1.019,1.078,0.996,1.90
21,meta-llama/Llama-3.2-1B,20,256,256,7.447,7.223,6.764,7.437,1.031,1.101,1.001,3.10
10,meta-llama/Llama-3.2-3B,20,256,256,14.065,13.718,12.849,14.091,1.025,1.095,0.998,2.54


In [82]:
# Train setting with BS=256 new_tokens=1
df[(df["batch_size"] == 128) & (df["prompt_len"] > 100) & (df["new_tokens"] == 1)]

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
51,HuggingFaceTB/SmolLM2-135M-Instruct,5382,128,1,0.913,0.866,0.680,0.850,1.055,1.342,1.074,5.49
50,Qwen/Qwen3-0.6B,5382,128,1,2.406,2.319,2.218,2.473,1.037,1.085,0.973,3.74
54,Qwen/Qwen3-1.7B,5382,128,1,4.479,4.332,3.957,4.480,1.034,1.132,1.000,3.39
64,meta-llama/Llama-3.1-8B,5382,128,1,13.146,12.851,11.545,13.006,1.023,1.139,1.011,2.30
37,meta-llama/Llama-3.2-1B,5382,128,1,2.600,2.516,2.310,2.616,1.034,1.126,0.994,3.35
52,meta-llama/Llama-3.2-3B,5382,128,1,6.907,6.692,6.152,6.909,1.032,1.123,1.000,3.20


In [83]:
# On device decode with BS=1 new_tokens=256
df[((df["batch_size"] == 1) | (df["batch_size"] == 64)) & (df["new_tokens"] == 256)]

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
22,Qwen/Qwen3-0.6B,20,1,256,3.029,2.815,2.529,3.024,1.076,1.197,1.002,7.60
13,Qwen/Qwen3-0.6B,20,64,256,4.821,4.585,4.215,4.917,1.051,1.144,0.980,5.14
1,Qwen/Qwen3-1.7B,20,1,256,2.139,1.998,1.800,2.184,1.070,1.188,0.979,7.02
8,Qwen/Qwen3-1.7B,20,64,256,2.993,2.868,2.618,3.034,1.043,1.143,0.986,4.35
57,meta-llama/Llama-3.1-8B,20,1,256,6.945,6.462,5.900,6.831,1.075,1.177,1.017,7.48
68,meta-llama/Llama-3.1-8B,20,64,256,8.221,8.031,7.132,8.408,1.024,1.153,0.978,2.36
12,meta-llama/Llama-3.2-1B,20,1,256,2.062,1.931,1.751,2.066,1.068,1.178,0.998,6.77
34,meta-llama/Llama-3.2-1B,20,64,256,3.065,2.892,2.654,3.071,1.060,1.155,0.998,5.99
28,meta-llama/Llama-3.2-3B,20,1,256,4.221,3.947,3.615,4.320,1.069,1.168,0.977,6.92
17,meta-llama/Llama-3.2-3B,20,64,256,6.046,5.646,5.148,6.068,1.071,1.175,0.996,7.09


In [84]:
df[(df["model_id"] == "meta-llama/Llama-3.1-8B")]

,model_id,prompt_len,batch_size,new_tokens,REF,COUPLED,NONORM,REFREF,speedup,nonorm,refref,speedup_pct
58,meta-llama/Llama-3.1-8B,20,1,1,0.027,0.027,0.024,0.029,1.015,1.136,0.934,1.52
63,meta-llama/Llama-3.1-8B,20,1,64,1.686,1.645,1.481,1.703,1.025,1.138,0.990,2.50
57,meta-llama/Llama-3.1-8B,20,1,256,6.945,6.462,5.900,6.831,1.075,1.177,1.017,7.48
56,meta-llama/Llama-3.1-8B,20,64,1,0.067,0.066,0.065,0.067,1.009,1.021,0.990,0.87
62,meta-llama/Llama-3.1-8B,20,64,64,1.995,1.946,1.687,1.976,1.025,1.183,1.010,2.52
68,meta-llama/Llama-3.1-8B,20,64,256,8.221,8.031,7.132,8.408,1.024,1.153,0.978,2.36
67,meta-llama/Llama-3.1-8B,20,256,1,0.206,0.203,0.197,0.208,1.016,1.047,0.993,1.63
66,meta-llama/Llama-3.1-8B,20,256,64,4.189,4.100,3.848,4.159,1.022,1.089,1.007,2.15
60,meta-llama/Llama-3.1-8B,20,256,256,18.251,17.910,16.924,18.318,1.019,1.078,0.996,1.90
59,meta-llama/Llama-3.1-8B,5382,1,1,0.134,0.133,0.124,0.133,1.010,1.077,1.007,0.95


In [85]:
df_plot = df[(df["new_tokens"] == 64)]
print(df_plot)

fig = px.line(df_plot, x="batch_size", y="speedup_pct", color="model_id")
fig.show()

df_plot = df[(df["new_tokens"] == 256) | (df["new_tokens"] == 64)]

fig = px.line(df_plot, x="batch_size", y="speedup_pct", color="model_id", line_dash="new_tokens")
fig.show()

                   model_id  prompt_len  batch_size  new_tokens    REF  \
29          Qwen/Qwen3-0.6B          20           1          64  0.750   
32          Qwen/Qwen3-0.6B          20          64          64  1.093   
15          Qwen/Qwen3-0.6B          20         256          64  2.312   
5           Qwen/Qwen3-1.7B          20           1          64  0.530   
4           Qwen/Qwen3-1.7B          20          64          64  0.704   
6           Qwen/Qwen3-1.7B          20         256          64  1.473   
63  meta-llama/Llama-3.1-8B          20           1          64  1.686   
62  meta-llama/Llama-3.1-8B          20          64          64  1.995   
66  meta-llama/Llama-3.1-8B          20         256          64  4.189   
25  meta-llama/Llama-3.2-1B          20           1          64  0.514   
24  meta-llama/Llama-3.2-1B          20          64          64  0.744   
30  meta-llama/Llama-3.2-1B          20         256          64  1.703   
23  meta-llama/Llama-3.2-3B          2

In [86]:
import kaleido
kaleido.get_chrome_sync()

PosixPath('/home/martin/space/coupled-rms-norm/.venv/lib/python3.13/site-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

In [87]:
df_plot = load_data(["multirun/decode-plot-0-25-normal-doubled/"])
df_plot = df_plot[(df_plot["batch_size"] != 8) & (df_plot["batch_size"] != 16)]

df_plot = df_plot.groupby(['batch_size', 'model_id'], as_index=False).max()

fig = px.line(
    df_plot, 
    x="batch_size", 
    y="speedup_pct",
    range_y=[0, None],
    range_x=[0, 256],
    color="model_id",
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set1,
    labels={
        "batch_size": "Batch Size",
        "speedup_pct": "Speedup (%)",
        "model_id": "Model Architecture"
    }
)

fig.update_layout(
    xaxis = dict(tickvals = list(range(0, 256+1, 64))),
)

paper_style_dict = {
    'layout.plot_bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.font.family': 'monospace', # 'Times New Roman',
    'layout.xaxis.linecolor': 'black',
    'layout.xaxis.ticks': 'outside',
    'layout.xaxis.mirror': True,
    'layout.xaxis.showline': True,
    'layout.xaxis.showgrid': True,
    'layout.xaxis.gridcolor': 'lightgray',
    'layout.xaxis.gridwidth': 0.1,
    'layout.yaxis.linecolor': 'black',
    'layout.yaxis.ticks': 'outside',
    'layout.yaxis.mirror': True,
    'layout.yaxis.showline': True,
    'layout.yaxis.showgrid': True,
    'layout.yaxis.gridcolor': 'lightgray',
    'layout.yaxis.gridwidth': 0.1,
    'layout.autosize': False,
    'layout.showlegend': True,
    'layout.legend.bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.legend.xanchor': 'right',
    'layout.legend.x': 1,
    'layout.legend.font.family': 'monospace',
}
fig.update(**paper_style_dict)

fig.write_image("decode-speedup.png", scale=4, width=800, height=500)
fig.write_image("decode-speedup.pdf")
fig.write_image("decode-speedup.svg", width=800, height=500)
fig.show()